In [ ]:
# --- Cell 1/5: setup --------------------------------------------------------
REPO_URL = "https://github.com/swaroopms658/loralink-reviewer.git"   # public
%cd /content
!rm -rf /content/loralink && git clone --depth 1 {REPO_URL} /content/loralink
%cd /content/loralink
!pip -q install -r loralink_reviewer_response/requirements-colab.txt
import sys; sys.path.insert(0, "/content/loralink")
# verify the patched sources against the committed SHA256SUMS (no --update -- a mismatch aborts the run)
!python loralink_reviewer_response/patch/checksums.py --verify

In [ ]:
# --- Cell 2/5: model download ---------------------------------------------
from huggingface_hub import snapshot_download
MODEL = "EleutherAI/gpt-neo-125M"
snapshot_download(MODEL, local_dir=f"./models/{MODEL}",
                  allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.bin",
                                  "merges.txt", "vocab.json", "tokenizer*"])

In [ ]:
# --- Cell 3/5: params (edit ACCOUNT_TAG and SHARD only) -----------------
ACCOUNT_TAG = "acct1"              # unique per Gmail account
SHARD       = ""  # the only knob besides ACCOUNT_TAG
WALL_BUDGET_MIN = 32
FAILED = 0                         # cells that raised; counted apart from DONE
import time; _NB_START = time.time()
def budget_left(): return WALL_BUDGET_MIN * 60 - (time.time() - _NB_START)

In [ ]:
# --- Cell 4/5: body -- delay x loss netem grid ---------------------------
from loralink_reviewer_response.cluster_launch import run_cluster

PER_RUN_ESTIMATE = 100
# SHARD may hold a delay subset like "0,25"; empty = all delays.
DELAYS = [int(x) for x in SHARD.split(",")] if SHARD.strip() else [0, 25, 50, 100]
LOSSES = [0, 1, 3]
PLANNED, DONE = len(DELAYS) * len(LOSSES), 0
csv = f"results_net_{ACCOUNT_TAG}.csv"

stop = False
for delay in DELAYS:
    if stop:
        break
    for loss in LOSSES:
        if budget_left() < PER_RUN_ESTIMATE:
            print("budget exhausted, stopping"); stop = True; break
        try:
            run_cluster(2, "wikitext", 0, model=MODEL, num_samples=20,
                        netem={"delay_ms": delay, "loss_pct": loss},
                        tag=f"net-d{delay}-l{loss}", results_csv=csv)
        except Exception as e:  # net-shim drop -> ConnectionError -> coordinator fails; skip
            print(f"  cell FAILED (delay={delay}ms loss={loss}%): {e}")
            FAILED += 1; continue
        DONE += 1
print(f"succeeded {DONE}/{PLANNED}, failed {FAILED}")


In [ ]:
# --- Cell 5/5: download ---------------------------------------------------
import json, glob, os
from google.colab import files

# main.py writes per-batch rows to results_<kind>_<tag>.csv and the single summary
# row to results_<kind>_<tag>.summary.csv (schemas differ) -- grab both.
produced = sorted(set(glob.glob(f"results_*_{ACCOUNT_TAG}.csv")
                      + glob.glob(f"results_*_{ACCOUNT_TAG}.summary.csv")))

json.dump({"tag": ACCOUNT_TAG, "shard": SHARD,
           "succeeded": DONE, "failed": FAILED, "planned": PLANNED,
           "result_files": [os.path.basename(f) for f in produced],
           "checksums": open("loralink_reviewer_response/patch/SHA256SUMS").read()},
          open(f"run_manifest_{ACCOUNT_TAG}.json", "w"), indent=2)

print(f"succeeded {DONE}/{PLANNED}, failed {FAILED}")
if not produced:
    print("NO RESULT FILES -- every run failed; check cell 4 output above. "
          "Downloading the manifest only.")
else:
    print("result files:", [os.path.basename(f) for f in produced])

for f in produced + [f"run_manifest_{ACCOUNT_TAG}.json"]:
    files.download(f)